In [ ]:
%pip install astropy
%pip install matplotlib
%pip install numpy
%pip install scipy
%pip install astroquery
%pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.0 MB/s eta 0:00:00


In [31]:
import sys
import os
import json
import requests
from urllib.parse import quote as urlencode
from astropy.io import fits
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt

In [32]:
def mast_query(request):
    request_url='https://mast.stsci.edu/api/v0/invoke'
    version = ".".join(map(str, sys.version_info[:3]))
    headers = {"Content-type": "application/x-www-form-urlencoded", "Accept": "text/plain", "User-agent":"python-requests/"+version}
    req_string = urlencode(json.dumps(request))
    resp = requests.post(request_url, data="request="+req_string, headers=headers)
    return resp.headers, resp.content.decode('utf-8')

def download_mast_fits_file(row):
    download_url = 'https://mast.stsci.edu/api/v0.1/Download/file?'
    out_dir = os.path.join("mastFiles", row['obs_collection'], row['obs_id'])
    if not os.path.exists(out_dir): os.makedirs(out_dir)
    filename = row.get('productFilename') or f"{row['obs_id']}.fits"
    out_path = os.path.join(out_dir, os.path.basename(filename))
    payload = {"uri": row.get('dataURI') or row.get('dataURL')}
    resp = requests.get(download_url, params=payload)
    with open(out_path,'wb') as f: f.write(resp.content)
    return out_path

In [33]:
def get_mast_data(object_name, radius=0.01):
    try:
        # Determine if it is a TIC ID or a standard name
        resolver_request = {'service':'Mast.Name.Lookup', 'params':{'input':object_name, 'format':'json'}}
        _, res_str = mast_query(resolver_request)
        res_json = json.loads(res_str)

        if not res_json.get('resolvedCoordinate'):
            return {'status': 'ERROR', 'msg': 'Could not resolve target name'}

        coords = res_json['resolvedCoordinate'][0]
        mast_request = {'service':'Mast.Caom.Cone', 'params':{'ra':coords['ra'], 'dec':coords['decl'], 'radius':radius}, 'format':'json'}
        _, data_str = mast_query(mast_request)
        return json.loads(data_str)
    except Exception as e:
        return {'status': 'ERROR', 'msg': str(e)}

In [35]:
mast_data = get_mast_data('Polaris')
mast_data

{'status': 'ERROR',
 'msg': "The transaction log for database 'tempdb' is full due to 'ACTIVE_TRANSACTION'."}

In [30]:
mast_data['data'][0]

KeyError: 'data'

In [ ]:
mast_data_with_data_url = [x for x in mast_data['data'] if 'dataURL' in x and x['dataURL'] is not None]
mast_data_with_data_url

[{'intentType': 'science',
  'obs_collection': 'TESS',
  'provenance_name': 'SPOC',
  'instrument_name': 'Photometer',
  'project': 'TESS',
  'filters': 'TESS',
  'wavelength_region': 'Optical',
  'target_name': '303256075',
  'target_classification': None,
  'obs_id': 'tess2019199201929-s0014-s0060-0000000303256075',
  's_ra': 37.9671985210327,
  's_dec': 89.2640510887674,
  'dataproduct_type': 'timeseries',
  'proposal_pi': 'Ricker, George',
  'calib_level': 3,
  't_min': 58815.57811289352,
  't_max': 59962.08441578704,
  't_exptime': 120,
  'em_min': 600,
  'em_max': 1000,
  'obs_title': None,
  't_obs_release': 59992,
  'proposal_id': 'N/A',
  'proposal_type': None,
  'sequence_number': 60,
  's_region': 'CIRCLE 37.96719852 89.26405109 0.00138889',
  'jpegURL': None,
  'dataURL': 'mast:TESS/product/tess2019199201929-s0014-s0060-0000000303256075-00692_dvt.fits',
  'dataRights': 'PUBLIC',
  'mtFlag': False,
  'srcDen': None,
  'obsid': 117285439,
  'distance': 0},
 {'intentType': 'sc

In [ ]:
mast_data_with_data_url[0]

{'intentType': 'science',
 'obs_collection': 'TESS',
 'provenance_name': 'SPOC',
 'instrument_name': 'Photometer',
 'project': 'TESS',
 'filters': 'TESS',
 'wavelength_region': 'Optical',
 'target_name': '303256075',
 'target_classification': None,
 'obs_id': 'tess2019199201929-s0014-s0060-0000000303256075',
 's_ra': 37.9671985210327,
 's_dec': 89.2640510887674,
 'dataproduct_type': 'timeseries',
 'proposal_pi': 'Ricker, George',
 'calib_level': 3,
 't_min': 58815.57811289352,
 't_max': 59962.08441578704,
 't_exptime': 120,
 'em_min': 600,
 'em_max': 1000,
 'obs_title': None,
 't_obs_release': 59992,
 'proposal_id': 'N/A',
 'proposal_type': None,
 'sequence_number': 60,
 's_region': 'CIRCLE 37.96719852 89.26405109 0.00138889',
 'jpegURL': None,
 'dataURL': 'mast:TESS/product/tess2019199201929-s0014-s0060-0000000303256075-00692_dvt.fits',
 'dataRights': 'PUBLIC',
 'mtFlag': False,
 'srcDen': None,
 'obsid': 117285439,
 'distance': 0}

In [19]:
fitUrl = download_mast_fits_file(mast_data_with_data_url[0])
print(f"Downloaded to: {fitUrl}")

Downloaded to: mastFiles/TESS/tess2019199201929-s0014-s0060-0000000303256075/303256075


In [9]:
import os
print(f"Current Working Directory: {os.getcwd()}")
print("Searching for all files in current directory...")
for root, dirs, files in os.walk('.'):
    for file in files:
        if not root.startswith('./.'):  # skip hidden folders
            print(os.path.join(root, file))

Current Working Directory: /content
Searching for all files in current directory...
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv


In [25]:
import json
import os
if os.path.exists('headers_counter.json'):
    with open('headers_counter.json', 'r') as f:
        stats = json.load(f)
    sorted_stats = sorted(stats.items(), key=lambda x: x[1], reverse=True)
    print(f"Total unique keywords: {len(sorted_stats)}")
    for keyword, count in sorted_stats[:20]:
        print(f"{keyword}: {count}")

Total unique keywords: 0


In [20]:
if 'fitUrl' in globals() and os.path.exists(fitUrl):
    with fits.open(fitUrl) as hdul:
        hdul.info()
        display(hdul[0].header[:30])
else:
    path_name = fitUrl if 'fitUrl' in globals() else 'unknown'
    print(f"File path {path_name} does not exist.")

Filename: mastFiles/TESS/tess2019199201929-s0014-s0060-0000000303256075/303256075
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      41   ()      
  1  TCE_1         1 BinTableHDU     92   165097R x 10C   [D, E, J, E, E, E, E, E, E, E]   
  2  TCE_2         1 BinTableHDU     92   165097R x 10C   [D, E, J, E, E, E, E, E, E, E]   
  3  Statistics    1 BinTableHDU    157   165097R x 38C   [D, E, J, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E]   


SIMPLE  =                    T / conforms to FITS standards                     
BITPIX  =                    8 / array data type                                
NAXIS   =                    0 / number of array dimensions                     
EXTEND  =                    T / file contains extensions                       
NEXTEND =                    4 / number of standard extensions                  
EXTNAME = 'PRIMARY '           / name of extension                              
EXTVER  =                    1 / extension version number (not format version)  
SIMDATA =                    F / file is based on simulated data                
ORIGIN  = 'NASA/Ames'          / institution responsible for creating this file 
DATE    = '2023-02-12'         / file creation date.                            
TSTART  =    1816.078913636710 / observation start time in BTJD                 
TSTOP   =    2962.585216527312 / observation stop time in BTJD                  
DATE-OBS= '2019-11-28T13:52:

In [29]:
import json
import os
from collections import Counter
from astropy.io import fits

def get_most_common_fits_keywords(target_names, max_objects=20):
    counter_file = 'headers_counter.json'
    overall_counts = Counter()
    if os.path.exists(counter_file):
        try:
            with open(counter_file, 'r') as f:
                overall_counts = Counter(json.load(f))
        except: pass

    processed_count = 0
    for name in target_names:
        if processed_count >= max_objects: break
        print(f"Processing: {name}...")
        # Use a radius of 0.02 to ensure we capture the specific TESS data products
        data = get_mast_data(name, radius=0.02)

        if not isinstance(data, dict) or 'data' not in data or not data['data']:
            print(f"  No data for {name}")
            continue

        # Filter for rows that have a dataURI/dataURL ending in .fits
        fits_rows = [r for r in data['data'] if str(r.get('dataURL') or r.get('dataURI') or '').lower().endswith('.fits')]
        if not fits_rows:
            print(f"  No FITS for {name}")
            continue

        try:
            local_path = download_mast_fits_file(fits_rows[0])
            if local_path and os.path.exists(local_path):
                with fits.open(local_path) as hdul:
                    for hdu in hdul:
                        overall_counts.update(hdu.header.keys())
                processed_count += 1
                print(f"  Success: {name}")
        except Exception as e: print(f"  Error {name}: {e}")

    with open(counter_file, 'w') as f:
        json.dump(dict(overall_counts), f, indent=4)
    print(f"\nFinished processing {processed_count} objects.")
    return overall_counts

# Target list for TESS processing
targets = ['TIC 303256075', 'TIC 27722169', 'TIC 27706396', 'TIC 150428135', 'TIC 14193351']
keyword_stats = get_most_common_fits_keywords(targets, max_objects=20)

Processing: TIC 303256075...
  No data for TIC 303256075
Processing: TIC 27722169...
  No data for TIC 27722169
Processing: TIC 27706396...
  No data for TIC 27706396
Processing: TIC 150428135...
  No data for TIC 150428135
Processing: TIC 14193351...
  No data for TIC 14193351

Finished processing 0 objects.
